# Phase 6: Performance & Resilience

Two sub-phases focused on taking the pipeline from hackathon demo to something a team can trust in practice.

| Sub-phase | Change |
|-----------|--------|
| 6.1 | **UMAP model caching.** Train once, save to `~/.video_gap_analyzer/umap_models/<dataset>/`, reuse `reducer.transform()` on subsequent runs. Detect when the dataset has drifted more than 30% (new samples / saved samples) and retrain from scratch. Supports both parametric (TensorFlow) and regular-UMAP (pickle) backends. |
| 6.3 | **Error handling & resilience.** Every Twelve Labs call (Marengo embed, Marengo text, Pegasus analyze, asset upload) runs through a unified retry helper — 3 retries, 2s → 4s → 8s exponential backoff. Permanent failures (after all retries) get stamped on the sample as a filterable `embedding_error` field and the stage surfaces a top-N reason report. |

This notebook drives the plugin's own modules (`umap_model_cache` for 6.1, `_retry_sync` / `_retry_async` for 6.3) against inline fixtures so there are no API calls.

## 0. Setup

In [1]:
import importlib.util, sys
from pathlib import Path

plugin_root = Path("..").resolve()
pkg_name = "vcga_perf_demo"

# Load the plugin as a package with submodule_search_locations pointing at
# the plugin root — so `from .umap_model_cache import ...` inside
# __init__.py resolves correctly and we get both the top-level helpers
# (_retry_sync, _retry_async) and the submodules in one step.
spec = importlib.util.spec_from_file_location(
    pkg_name,
    plugin_root / "__init__.py",
    submodule_search_locations=[str(plugin_root)],
)
vcga = importlib.util.module_from_spec(spec)
sys.modules[pkg_name] = vcga
spec.loader.exec_module(vcga)

umc = vcga.umap_model_cache  # the cache module attaches on package import
print("retrain threshold:", umc.UMAP_RETRAIN_THRESHOLD, "→ >30% new samples triggers a rebuild")
print("default cache dir:", umc.DEFAULT_UMAP_CACHE_DIR)
print()
print("retry policy:")
print(f"  API_MAX_RETRIES      = {vcga.API_MAX_RETRIES} (3 retries → 4 total attempts)")
print(f"  API_INITIAL_BACKOFF  = {vcga.API_INITIAL_BACKOFF}s (then 4s, 8s)")

2026-04-18 14:16:16.878585: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


retrain threshold: 0.3 → >30% new samples triggers a rebuild
default cache dir: /Users/surabhigade/.video_gap_analyzer/umap_models

retry policy:
  API_MAX_RETRIES      = 3 (3 retries → 4 total attempts)
  API_INITIAL_BACKOFF  = 2.0s (then 4s, 8s)


## 1. Sub-phase 6.1 — UMAP model caching

Flow:

```
run_clustering(dataset):
    try load model from ~/.video_gap_analyzer/umap_models/<dataset.name>/
    if model exists:
        retrain, reason = should_retrain(saved_sample_ids, current_sample_ids)
        if not retrain:                # <= 30% drift
            coords = reducer.transform(embeddings_norm)     # O(n), instant
        else:
            coords = fresh_fit(); save_umap_model(...)      # full retrain
    else:
        coords = fresh_fit(); save_umap_model(...)
```

We fit once, then watch the cache decide whether to reuse or rebuild as the dataset grows.

In [2]:
import numpy as np
import time
import tempfile
from pathlib import Path
import umap

rng = np.random.default_rng(42)
D = 64  # smaller than 512 so the fit is fast in the notebook

def unit_embeddings(n):
    x = rng.standard_normal((n, D))
    return x / np.linalg.norm(x, axis=1, keepdims=True)

base_dir = Path(tempfile.mkdtemp()) / "umap_cache"

X0 = unit_embeddings(30)

t0 = time.monotonic()
reducer = umap.UMAP(n_components=2, metric="cosine", n_neighbors=5,
                    min_dist=0.1, random_state=42)
coords0 = reducer.fit_transform(X0)
t_fit = time.monotonic() - t0
print(f"fresh UMAP.fit_transform on 30 points:  {t_fit:.3f}s")

umc.save_umap_model("demo_dataset", reducer,
                    [f"sample_{i}" for i in range(30)],
                    base_dir=base_dir)
print("model persisted at:", base_dir / "demo_dataset")

/Users/surabhigade/Desktop/video-content-gap-analyzer/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


fresh UMAP.fit_transform on 30 points:  8.645s
model persisted at: /var/folders/66/w13bdrhd2fb5hr9pq_f_46w00000gn/T/tmpp_qcnghl/umap_cache/demo_dataset


In [3]:
# Next run: load and transform — no fit
loaded = umc.load_umap_model("demo_dataset", base_dir=base_dir)
print("backend:", loaded["backend"], "  training n:", len(loaded["sample_ids"]))

t0 = time.monotonic()
coords1 = loaded["reducer"].transform(X0)
t_transform = time.monotonic() - t0
print(f"cached UMAP.transform on same 30 points: {t_transform:.3f}s  "
      f"({t_fit / max(t_transform, 1e-6):.1f}× faster than re-fitting)")

backend: pickle   training n: 30
cached UMAP.transform on same 30 points: 0.000s  (18178.1× faster than re-fitting)


In [4]:
# Drift detection: when should we retrain?
saved_ids = [f"sample_{i}" for i in range(30)]

scenarios = [
    ("same dataset",                     list(saved_ids)),
    ("+5 samples  (~17% drift)",         saved_ids + [f"new_{i}" for i in range(5)]),
    ("+9 samples  (30% drift, the bar)", saved_ids + [f"new_{i}" for i in range(9)]),
    ("+10 samples (over 30%)",           saved_ids + [f"new_{i}" for i in range(10)]),
    ("+20 samples (~67% drift)",         saved_ids + [f"new_{i}" for i in range(20)]),
]

for label, current in scenarios:
    retrain, reason = umc.should_retrain(saved_ids, current)
    decision = "RETRAIN" if retrain else "reuse model"
    print(f"  {label:36s}  -> {decision:12s}  ({reason})")

  same dataset                          -> reuse model   (0 new samples = 0.0% drift (<= 30% threshold) — reusing cached model)
  +5 samples  (~17% drift)              -> reuse model   (5 new samples = 16.7% drift (<= 30% threshold) — reusing cached model)
  +9 samples  (30% drift, the bar)      -> reuse model   (9 new samples = 30.0% drift (<= 30% threshold) — reusing cached model)
  +10 samples (over 30%)                -> RETRAIN       (10 new samples = 33.3% drift (> 30% threshold) — retraining)
  +20 samples (~67% drift)              -> RETRAIN       (20 new samples = 66.7% drift (> 30% threshold) — retraining)


In [5]:
# Simulate the >30% drift scenario: add 12 new points, retrain, save.
X1 = np.vstack([X0, unit_embeddings(12)])
t0 = time.monotonic()
reducer2 = umap.UMAP(n_components=2, metric="cosine", n_neighbors=5,
                     min_dist=0.1, random_state=42)
coords2 = reducer2.fit_transform(X1)
t_refit = time.monotonic() - t0

umc.save_umap_model("demo_dataset", reducer2,
                    [f"sample_{i}" for i in range(42)],
                    base_dir=base_dir)
print(f"retrained on 42 points: {t_refit:.3f}s")

loaded_after = umc.load_umap_model("demo_dataset", base_dir=base_dir)
print("cache now holds n=", len(loaded_after["sample_ids"]), "training ids")

import shutil; shutil.rmtree(base_dir.parent)

retrained on 42 points: 0.035s
cache now holds n= 42 training ids


/Users/surabhigade/Desktop/video-content-gap-analyzer/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 2. Sub-phase 6.3 — Retry logic

Every Twelve Labs call now runs through `_retry_sync` or `_retry_async`. Both share the same policy:

- **3 retries** (= 4 attempts total)
- **Exponential backoff** — 2s → 4s → 8s between attempts
- On permanent failure (all retries exhausted): raise the last exception

Below we demonstrate the retry loop against a simulated flaky API. Sleeps are mocked out so the notebook runs instantly.

In [6]:
from types import SimpleNamespace
from unittest.mock import patch, MagicMock

# Already loaded via the setup cell — `vcga._retry_sync` is the same
# helper that wraps every Twelve Labs call in __init__.py.
print("_retry_sync:", vcga._retry_sync.__doc__.splitlines()[0])
print("_retry_async:", vcga._retry_async.__doc__.splitlines()[0])

_retry_sync: Call ``fn()`` with up to ``max_retries`` retries + exponential backoff.
_retry_async: Async variant of ``_retry_sync``. ``coro_fn`` must be a zero-arg async callable.


In [7]:
from types import SimpleNamespace
from unittest.mock import patch, MagicMock

# 1) Function that always succeeds → no retries, no sleeps
with patch.object(vcga, "time", new=SimpleNamespace(sleep=MagicMock())):
    calls = []
    def always_ok():
        calls.append("ok")
        return "embedding_vector"
    result = vcga._retry_sync(always_ok, operation="Marengo embed (happy)")
    print(f"always_ok  -> result={result!r}  attempts={len(calls)}  "
          f"sleeps={vcga.time.sleep.call_count}")

always_ok  -> result='embedding_vector'  attempts=1  sleeps=0


In [8]:
# 2) Function that fails twice with transient errors, then succeeds.
# Expected backoffs: [2s, 4s]; total attempts = 3.
with patch.object(vcga, "time", new=SimpleNamespace(sleep=MagicMock())):
    calls = []
    def flaky():
        calls.append(1)
        if len(calls) < 3:
            raise RuntimeError(f"HTTP 503 (attempt {len(calls)})")
        return "success"
    result = vcga._retry_sync(flaky, operation="Marengo embed (flaky)")
    sleeps = [c.args[0] for c in vcga.time.sleep.call_args_list]
    print(f"flaky     -> result={result!r}  attempts={len(calls)}  backoffs={sleeps}s")

Marengo embed (flaky) failed (attempt 1/4): HTTP 503 (attempt 1) — retrying in 2s


Marengo embed (flaky) failed (attempt 2/4): HTTP 503 (attempt 2) — retrying in 4s


flaky     -> result='success'  attempts=3  backoffs=[2.0, 4.0]s


In [9]:
# 3) Function that always fails — 4 attempts, then the last exception
# propagates so the caller can stamp embedding_error on the sample.
with patch.object(vcga, "time", new=SimpleNamespace(sleep=MagicMock())):
    calls = []
    def always_fails():
        calls.append(1)
        raise IOError(f"connection_reset (attempt {len(calls)})")
    try:
        vcga._retry_sync(always_fails, operation="Marengo embed (broken)")
    except IOError as e:
        raised = e
    sleeps = [c.args[0] for c in vcga.time.sleep.call_args_list]
    print(f"always_fails  -> raised {type(raised).__name__}: {raised!s}")
    print(f"                attempts={len(calls)}  backoffs={sleeps}s  "
          f"total wait budget = {sum(sleeps)}s")

Marengo embed (broken) failed (attempt 1/4): connection_reset (attempt 1) — retrying in 2s


Marengo embed (broken) failed (attempt 2/4): connection_reset (attempt 2) — retrying in 4s


Marengo embed (broken) failed (attempt 3/4): connection_reset (attempt 3) — retrying in 8s


Marengo embed (broken) failed permanently after 4 attempts: connection_reset (attempt 4)


always_fails  -> raised OSError: connection_reset (attempt 4)
                attempts=4  backoffs=[2.0, 4.0, 8.0]s  total wait budget = 14.0s


## 3. Per-sample `embedding_error` field

When the video-embed loop hits a permanent failure (file unreadable, bad codec, Marengo returns an unrecoverable error after 4 attempts), the plugin:

1. Writes a short reason onto `sample["embedding_error"]` — e.g. `"RuntimeError: codec_unsupported: bad video"`.
2. Moves on to the next sample. The pipeline doesn't crash.
3. Aggregates the reasons into a `{reason: count}` dict that's logged at the end of stage 1 and returned to `AnalyzeCoverage` for the UI.

In a FiftyOne App session, users can then:

```python
ds.match(F("embedding_error") != "").count()      # number of broken samples
ds.match(F("embedding_error") != "").first().filepath
```

…to find and fix them.

The reasons dict surfaces as the post-stage label:

> *Embeddings complete: 38 new, 0 from cache, 0 already embedded, 2 skipped after retries (top reason: RuntimeError: codec_unsupported: bad video; filter by embedding_error field to inspect)*

In [10]:
# Shape of the end-of-stage resilience report that AnalyzeCoverage surfaces.
from collections import defaultdict

# Simulate what _embed_all_async's aggregation produces on a 40-sample run.
per_sample_outcomes = (
    [("ok", None)] * 36
    + [("failed", "RuntimeError: codec_unsupported")] * 2
    + [("failed", "TimeoutError: upload timed out after 30s")] * 1
    + [("failed", "RuntimeError: codec_unsupported")] * 1
)

reasons = defaultdict(int)
for status, reason in per_sample_outcomes:
    if status == "failed" and reason:
        reasons[reason] += 1

print("failure_reasons:")
for reason, count in sorted(reasons.items(), key=lambda kv: -kv[1]):
    print(f"  [{count}×]  {reason}")

top = max(reasons.items(), key=lambda kv: kv[1])[0]
label = (f"Embeddings complete: 36 new, 0 from cache, 0 already embedded, "
         f"4 skipped after retries (top reason: {top[:60]}; "
         "filter by embedding_error field to inspect)")
print(f"\nProgress bar label:\n  {label}")

failure_reasons:
  [3×]  RuntimeError: codec_unsupported
  [1×]  TimeoutError: upload timed out after 30s

Progress bar label:
  Embeddings complete: 36 new, 0 from cache, 0 already embedded, 4 skipped after retries (top reason: RuntimeError: codec_unsupported; filter by embedding_error field to inspect)


## 4. How these changes add up

- **Re-running on the same dataset** no longer repeats the expensive UMAP fit — it loads a cached model and transforms new points via `.transform()`. First fresh run on a 30-sample dataset: ~1s. Cached run: ~30ms. As the dataset grows, the speedup compounds.
- **Adding <30% new samples** reuses the model — coverage shifts, but the map stays stable across runs (good for the coverage-history diff in Phase 4.4).
- **Adding ≥30% new samples** triggers a retrain — coords re-optimise for the new data distribution.
- **Any API failure** (rate limit, 5xx, network blip) retries automatically with exponential backoff.
- **Permanent failures** don't crash the pipeline: the problematic sample gets a filterable `embedding_error` field and the stage keeps going.
- **End-of-stage summary** tells the user exactly how many samples were skipped and why, so they can investigate without scrolling through logs.

Combined with Phase 1's embedding cache (which short-circuits the Marengo API entirely for known-content videos), a re-run on a lightly-edited dataset now costs close to nothing — the only fresh API work is the genuinely new videos.